In [2]:
import pandas as pd
import numpy as np
import nltk
import spacy
import gensim
import warnings
warnings.filterwarnings("ignore")

In [3]:
# DATA IMPORTING 

In [4]:
test_df = pd.read_csv(
    r"C:\Users\Lenovo\Desktop\github projects\nlp rnn assingment\data train test validate\test.txt",
    sep=";",
    names=["text", "emotion"]
)
train_df = pd.read_csv(
    r"C:\Users\Lenovo\Desktop\github projects\nlp rnn assingment\data train test validate\train.txt",
    sep=";",
    names=["text", "emotion"]
)
val_df = pd.read_csv(
    r"C:\Users\Lenovo\Desktop\github projects\nlp rnn assingment\data train test validate\val.txt",
    sep=";",
    names=["text", "emotion"]
)

In [5]:
print(test_df.head())
print("\n")
print(train_df.head())
print("\n")
print(train_df.head())

                                                text  emotion
0  im feeling rather rotten so im not very ambiti...  sadness
1          im updating my blog because i feel shitty  sadness
2  i never make her separate from me because i do...  sadness
3  i left with my bouquet of red and yellow tulip...      joy
4    i was feeling a little vain when i did this one  sadness


                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger


                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i

In [6]:
 # checking for missing values
for name,df in [("Train",train_df),("Test",test_df),("valadition", val_df)]:
    print(f"\n{name} Dataset")
    print(df.isnull().sum())


Train Dataset
text       0
emotion    0
dtype: int64

Test Dataset
text       0
emotion    0
dtype: int64

valadition Dataset
text       0
emotion    0
dtype: int64


In [7]:
# checking for duplicate values 

In [8]:
for name ,df in [("Train",train_df),("Test",test_df),("validate",val_df)]:
    print(f"\n{name} Dataset")
    print(df.duplicated().sum())


Train Dataset
1

Test Dataset
0

validate Dataset
0


In [9]:
# Basic Preprocessing 
import re 

def clean_text(text):
    text=text.lower()

    text=re.sub(r'http\S+|www\S+','',text)

    text=re.sub(r'<.*?>', '', text)

    text=re.sub(r'[^a-zA-Z\s]', '', text)

    text=re.sub(r'\s+',' ',text).strip()

    return text


In [10]:
train_df["clean_text"]=train_df["text"].apply(clean_text)
test_df["clean_text"]=test_df["text"].apply(clean_text)
val_df["clean_text"]=val_df["text"].apply(clean_text)

In [11]:
for head in [ train_df,test_df,val_df]:
    print(display(head[["text","clean_text"]].head()))
    print("--"*50)

,text,clean_text
0,i didnt feel humiliated,i didnt feel humiliated
1,i can go from feeling so hopeless to so damned...,i can go from feeling so hopeless to so damned...
2,im grabbing a minute to post i feel greedy wrong,im grabbing a minute to post i feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...,i am ever feeling nostalgic about the fireplac...
4,i am feeling grouchy,i am feeling grouchy


None
----------------------------------------------------------------------------------------------------


,text,clean_text
0,im feeling rather rotten so im not very ambiti...,im feeling rather rotten so im not very ambiti...
1,im updating my blog because i feel shitty,im updating my blog because i feel shitty
2,i never make her separate from me because i do...,i never make her separate from me because i do...
3,i left with my bouquet of red and yellow tulip...,i left with my bouquet of red and yellow tulip...
4,i was feeling a little vain when i did this one,i was feeling a little vain when i did this one


None
----------------------------------------------------------------------------------------------------


,text,clean_text
0,im feeling quite sad and sorry for myself but ...,im feeling quite sad and sorry for myself but ...
1,i feel like i am still looking at a blank canv...,i feel like i am still looking at a blank canv...
2,i feel like a faithful servant,i feel like a faithful servant
3,i am just feeling cranky and blue,i am just feeling cranky and blue
4,i can have for a treat or if i am feeling festive,i can have for a treat or if i am feeling festive


None
----------------------------------------------------------------------------------------------------


#### Tokenization 

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [13]:
tokenizer=Tokenizer(num_words=10000,oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["clean_text"])

In [14]:
X_train=tokenizer.texts_to_sequences(train_df["clean_text"])
X_test=tokenizer.texts_to_sequences(test_df["clean_text"])
X_val=tokenizer.texts_to_sequences(val_df["clean_text"])

In [15]:
# Label Encododint(Target Variabel)
from sklearn.preprocessing import LabelEncoder

In [16]:
label_encoder=LabelEncoder()

In [17]:
y_train=label_encoder.fit_transform(train_df["emotion"])
y_test=label_encoder.transform(test_df["emotion"])
y_val=label_encoder.transform(val_df["emotion"])

In [18]:
label_encoder.classes_

array(['anger', 'fear', 'joy', 'love', 'sadness', 'surprise'],
      dtype=object)

In [19]:
#### Padding 
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [20]:
max_length=60

X_train=pad_sequences(X_train,maxlen=max_length,padding='post',truncating='post')
X_test=pad_sequences(X_test,maxlen=max_length,padding='post',truncating='post')
X_val=pad_sequences(X_val,maxlen=max_length,padding='post',truncating='post')

# MODELLING

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding,SimpleRNN,LSTM,GRU,Dense,Dropout,Bidirectional)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (classification_report,confusion_matrix,accuracy_score)

In [22]:
vocab_size=len(tokenizer.word_index)+1

max_length=X_train.shape[1]

num_classes=len(label_encoder.classes_)

embedding_dim=128


# EVALUATION FUNCTION 

In [23]:
def evaluate_model(model,X_train,y_train,X_test,y_test):
    # Train Predictions 
    train_pred=model.predict(X_train,verbose=0)
    train_pred_classes=np.argmax(train_pred,axis=1)

    #Test Predictions
    test_pred=model.predict(X_test,verbose=0)
    test_pred_classes=np.argmax(test_pred,axis=1)

    #ACCURACY
    train_acc=accuracy_score(y_train,train_pred_classes)
    test_acc=accuracy_score(y_test,test_pred_classes)

    #cofusion Matrix 
    cm=confusion_matrix(y_test,test_pred_classes)
    #classification_report 
    report=classification_report(y_test,test_pred_classes)

    print("MODEL EVALUATION")
    print("="*50)

    print(f"\n Train Accuracy :{train_acc}")
    print(f"\n Test Accuracy :{test_acc}")

    print("\n Classification Report")
    print(report)

    print("\n Confusion Matrix")
    print(cm)
    
    
    
    



### 1.Simple RNN

In [24]:
rnn_model=Sequential([Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_length),
                      SimpleRNN(128),Dense(num_classes,activation='softmax')])

rnn_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
rnn_model.build(input_shape=(None, max_length))

rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 60, 128)        │     1,947,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,980,678 (7.56 MB)

 Trainable params: 1,980,678 (7.56 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
history_rnn=rnn_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 54s 78ms/step - accuracy: 0.3206 - loss: 1.5911 - val_accuracy: 0.3520 - val_loss: 1.5912
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 39s 77ms/step - accuracy: 0.3271 - loss: 1.5823 - val_accuracy: 0.3520 - val_loss: 1.5826
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 32s 64ms/step - accuracy: 0.3271 - loss: 1.5824 - val_accuracy: 0.3520 - val_loss: 1.6088
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 26s 52ms/step - accuracy: 0.3291 - loss: 1.5785 - val_accuracy: 0.3520 - val_loss: 1.5821
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 69ms/step - accuracy: 0.3294 - loss: 1.5801 - val_accuracy: 0.3520 - val_loss: 1.5821
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 36s 72ms/step - accuracy: 0.3284 - loss: 1.5903 - val_accuracy: 0.2760 - val_loss: 1.6341
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 69ms/step - accuracy: 0.3207 - loss: 1.5957 - val_accuracy: 0.3455 - val_loss: 1.5849
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 41s 81ms/step - accuracy: 0.3190 - loss: 1.5903 - 

In [26]:
evaluate_model(rnn_model,X_train,y_train,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.335625

 Test Accuracy :0.3475

 Classification Report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       275
           1       0.00      0.00      0.00       224
           2       0.35      1.00      0.52       695
           3       0.00      0.00      0.00       159
           4       0.00      0.00      0.00       581
           5       0.00      0.00      0.00        66

    accuracy                           0.35      2000
   macro avg       0.06      0.17      0.09      2000
weighted avg       0.12      0.35      0.18      2000


 Confusion Matrix
[[  0   1 274   0   0   0]
 [  0   0 224   0   0   0]
 [  0   0 695   0   0   0]
 [  0   0 159   0   0   0]
 [  0   0 581   0   0   0]
 [  0   0  66   0   0   0]]


### 2. LSTM

In [27]:
lstm_model=Sequential([Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_length),LSTM(128),
                       Dense(num_classes,activation='softmax')])
lstm_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
lstm_model.build(input_shape=(None, max_length))
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 60, 128)        │     1,947,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,079,366 (7.93 MB)

 Trainable params: 2,079,366 (7.93 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
history_lstm=lstm_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 97s 146ms/step - accuracy: 0.3313 - loss: 1.5828 - val_accuracy: 0.3520 - val_loss: 1.5803
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 62s 124ms/step - accuracy: 0.3316 - loss: 1.5790 - val_accuracy: 0.3520 - val_loss: 1.5807
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 91s 142ms/step - accuracy: 0.3320 - loss: 1.5765 - val_accuracy: 0.3520 - val_loss: 1.5820
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 72s 144ms/step - accuracy: 0.3360 - loss: 1.5760 - val_accuracy: 0.3520 - val_loss: 1.5819
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 69s 137ms/step - accuracy: 0.3316 - loss: 1.5737 - val_accuracy: 0.3525 - val_loss: 1.5456
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 76s 152ms/step - accuracy: 0.3405 - loss: 1.5507 - val_accuracy: 0.3590 - val_loss: 1.5633
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 72s 132ms/step - accuracy: 0.3402 - loss: 1.5632 - val_accuracy: 0.3520 - val_loss: 1.5803
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 49s 97ms/step - accuracy: 0.3318 - loss: 1.

In [29]:
evaluate_model(lstm_model,X_train,y_train,X_test,y_test)

MODEL EVALUATION

 Train Accuracy :0.329625

 Test Accuracy :0.3315

 Classification Report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       275
           1       0.19      0.61      0.29       224
           2       0.41      0.76      0.53       695
           3       0.00      0.00      0.00       159
           4       0.00      0.00      0.00       581
           5       0.00      0.00      0.00        66

    accuracy                           0.33      2000
   macro avg       0.10      0.23      0.14      2000
weighted avg       0.16      0.33      0.22      2000


 Confusion Matrix
[[  0 121 154   0   0   0]
 [  0 136  88   0   0   0]
 [  0 168 527   0   0   0]
 [  0  72  87   0   0   0]
 [  0 171 410   0   0   0]
 [  0  33  33   0   0   0]]


In [34]:
cd "C:\Users\Lenovo\Desktop\github projects\nlp rnn assingment"

C:\Users\Lenovo\Desktop\github projects\nlp rnn assingment


In [35]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Emotion_Classification_using_DL.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


### 3. GRU

In [ ]:
gru_model=Sequential([Embedding(vicab_size,embedding_dim,input_length=max_length),GRU(128)
                      Dense(num_classes,activation='softmax')])
gru_model.compile(optimizer='adam',loss='sparse_categorical_entropy',metrics=['accuracy'])



In [ ]:
history_gru=gru_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

In [ ]:
evaluate_model(gru_model,X_train,y_train,X_test,y_test)

### 4. Bi-Directional RNN

In [ ]:
birnn_model=Squential([Embedding(vocab_size,embedding_dim,input_length=max_length),BiDirectional(SimpleRNN(128)),
                       Dense(num_classes,activation='softmax')])

birnn_model.compile(optimizer='adam',loss='sparse_categorical_entropy',metrics=['accuracy'])

                                                                                                 

In [ ]:
history_birnn=birnn_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

In [ ]:
evaluate_model(birnn_model,X_train,y_train,X_test,y_test)

### 5. BI-Directional LSTM

In [ ]:
bilstm_model = Sequential([

    Embedding(vocab_size,
              embedding_dim,
              input_length=max_length),

    Bidirectional(
        LSTM(128)
    ),

    Dense(num_classes,
          activation='softmax')
])

In [ ]:
bilstm_model.compile(optimizer='adam',loss='sparse_categorical_entropy',metrics=['accuracy'])

In [ ]:
history_bilstm=bilstm_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32)

In [ ]:
evaluate_modelevaluate_model(bilstm_model,X_train,y_train,X_test,y_test)

### 6. Bi Directional GRU